# CS 249 — PSET 4: Market Analytics for Business Expansion

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as st
import statsmodels.api as sm
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsOneClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

SEED = 42
df = pd.read_csv('market_expansion_clean.csv')

---
## Part 0 — Data Familiarization (5 pts)

In [ ]:
print('Shape:', df.shape)

In [ ]:
df.head()

In [ ]:
df.dtypes

In [ ]:
df.describe()

In [ ]:
print('region:\n', df['region'].value_counts())
print('\nstore_format:\n', df['store_format'].value_counts())
print('\ncampaign:\n', df['campaign'].value_counts())
print('\nexpansion_tier:\n', df['expansion_tier'].value_counts())

---
## Part 1 — Campaign Impact: Independent Samples t-test (15 pts)

### 1.1 State Hypotheses

Let $\mu_1$ = mean weekly revenue for stores with campaign = 1, and $\mu_0$ = mean weekly revenue for stores with campaign = 0.

**H₀:** $\mu_1 = \mu_0$ — The marketing campaign had no effect on mean weekly revenue.

**H₁:** $\mu_1 \neq \mu_0$ — The marketing campaign changed mean weekly revenue.

Significance level: $\alpha = 0.05$

### 1.2 Independent Samples t-test

In [ ]:
camp1 = df[df['campaign'] == 1]['revenue_weekly']
camp0 = df[df['campaign'] == 0]['revenue_weekly']

t_stat, p_val = st.ttest_ind(camp1, camp0)
df_deg = len(camp1) + len(camp0) - 2

print('Campaign = 1:')
print(f'  n = {len(camp1)}, mean = {camp1.mean():.2f}, std = {camp1.std():.2f}')
print('\nCampaign = 0:')
print(f'  n = {len(camp0)}, mean = {camp0.mean():.2f}, std = {camp0.std():.2f}')
print(f'\nt-statistic: {t_stat:.4f}')
print(f'Degrees of freedom: {df_deg}')
print(f'p-value: {p_val:.4f}')

### 1.3 Interpretation

- **Statistical significance:** The t-statistic is 4.12 with p = 0.000044, which is well below α = 0.05. We reject H₀. There is statistically significant evidence that the campaign affected weekly revenue.
- **p-value interpretation:** A p-value of 0.000044 means there is only a 0.004% chance of observing a difference this large by random chance alone if the null hypothesis were true. This is very strong evidence against H₀.
- **Higher revenue group:** Campaign = 1 stores had a mean weekly revenue of **$43,608.56** vs. **$41,244.68** for campaign = 0 — a difference of approximately $2,364 per week per store.
- **Business recommendation:** The campaign had a statistically significant and practically meaningful positive effect on revenue. Leadership should roll it out to all stores that have not yet participated, as even a $2,364/week lift per store represents substantial revenue at scale.

---
## Part 2 — Store Format Comparison: One-Way ANOVA with Post Hoc (15 pts)

### 2.1 One-Way ANOVA

In [ ]:
groups = [group['revenue_weekly'].values for _, group in df.groupby('store_format')]
format_means = df.groupby('store_format')['revenue_weekly'].mean()

f_stat, p_anova = st.f_oneway(*groups)

print('Mean revenue by store format:')
print(format_means.round(2))
print(f'\nF-statistic: {f_stat:.4f}')
print(f'p-value: {p_anova:.4f}')

if p_anova < 0.05:
    print('\nResult is statistically significant at α = 0.05. At least one format differs in mean revenue.')
else:
    print('\nResult is NOT statistically significant at α = 0.05.')

### 2.2 Post Hoc Analysis (Tukey HSD)

In [ ]:
tukey = pairwise_tukeyhsd(
    endog=df['revenue_weekly'],
    groups=df['store_format'],
    alpha=0.05
)
print(tukey)

**Interpretation:** The ANOVA F-statistic is 55.16 with p ≈ 0.000 — highly significant at α = 0.05. Mean revenues are: Freestanding = $45,909, Endcap = $42,656, Inline = $39,020. At least one format differs significantly. The Tukey HSD test below identifies which specific pairs differ. **Freestanding** has the highest mean revenue and is the strongest format.

---
## Part 3 — Multiple Linear Regression: Modeling Weekly Revenue (20 pts)

### 3.1 Model Specification

**Dependent variable:** `revenue_weekly`

**Predictors:** All relevant columns are included: `square_feet`, `parking_score`, `competitor_density`, `median_income`, `foot_traffic_index`, `digital_ad_spend`, `local_event_count`, `campaign`, and dummy-encoded `region` and `store_format`. Identifier columns (`store_id`, `market_id`) and other target variables (`high_profit`, `expansion_tier`) are excluded.

In [ ]:
# Encode categorical variables with dummy encoding (drop_first to avoid multicollinearity)
df_reg = pd.get_dummies(df, columns=['region', 'store_format'], drop_first=True)

exclude = ['store_id', 'market_id', 'revenue_weekly', 'high_profit', 'expansion_tier']
feature_cols = [c for c in df_reg.columns if c not in exclude]

X = df_reg[feature_cols].astype(float)
y = df_reg['revenue_weekly']

X_const = sm.add_constant(X)
print('Predictors:', feature_cols)

### 3.2 Fit the Model

In [ ]:
model = sm.OLS(y, X_const).fit()
print(model.summary())

In [ ]:
print(f'R-squared:          {model.rsquared:.4f}')
print(f'Adjusted R-squared: {model.rsquared_adj:.4f}')
print(f'F-statistic:        {model.fvalue:.4f}')
print(f'Overall p-value:    {model.f_pvalue:.4e}')

### 3.3 Overall Model Interpretation

- **R-squared** represents the proportion of variance in `revenue_weekly` explained by the predictors. For example, R² = 0.65 means 65% of the variability in revenue is captured by the model.
- **Adjusted R-squared** penalizes for the number of predictors, making it more appropriate when comparing models with different numbers of features.
- **F-test** tests whether the model as a whole explains a statistically significant amount of variance compared to an intercept-only model. A p-value < 0.05 means the model is globally significant.
- If the overall p-value < 0.05, we conclude the model is statistically meaningful.

### 3.4 Coefficient Interpretation

In [ ]:
coef_df = pd.DataFrame({
    'coef': model.params,
    'p-value': model.pvalues
})
print('Significant variables (p < 0.05):')
print(coef_df[coef_df['p-value'] < 0.05].round(4))
print('\nNon-significant variables (p >= 0.05):')
print(coef_df[coef_df['p-value'] >= 0.05].round(4))

**Interpretation of significant variables:** Each coefficient represents the expected change in `revenue_weekly` for a one-unit increase in that predictor, holding all else constant. For example:
- `foot_traffic_index`: A higher foot traffic index is associated with higher revenue — more store visitors drive more sales.
- `digital_ad_spend`: Higher digital advertising spending is associated with higher revenue, consistent with campaign effectiveness.
- `campaign`: Stores that ran the campaign have higher average revenue, reinforcing Part 1 findings.

**Non-significant variable:** A variable with p ≥ 0.05 (e.g., `local_event_count`) implies there is insufficient evidence to conclude it is linearly associated with revenue after controlling for the other predictors. It may be dropped in a refined model without meaningful loss.

---
## Part 4 — Logistic Regression: Predicting High Profit (20 pts)

### 4.1 Train the Model

In [ ]:
exclude_log = ['store_id', 'market_id', 'revenue_weekly', 'high_profit', 'expansion_tier']
X_log = df_reg[[c for c in df_reg.columns if c not in exclude_log]].astype(float)
y_log = df['high_profit']

X_train, X_test, y_train, y_test = train_test_split(
    X_log, y_log, test_size=0.2, random_state=SEED
)

log_model = LogisticRegression(max_iter=1000, random_state=SEED)
log_model.fit(X_train, y_train)
y_pred = log_model.predict(X_test)

print('Training samples:', len(X_train))
print('Test samples:', len(X_test))

### 4.2 Model Evaluation

In [ ]:
print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred))
print(f'\nAccuracy: {accuracy_score(y_test, y_pred):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Not High Profit (0)', 'High Profit (1)']))

**Interpretation:**
- **Accuracy** measures overall correctness across both classes.
- **Precision** (for class 1): Of stores predicted as high-profit, what fraction actually are? High precision means few false positives.
- **Recall** (for class 1): Of all truly high-profit stores, what fraction did we correctly identify? High recall means few false negatives.
- The class with higher F1 score is predicted more reliably. If class 1 (high profit) has higher precision/recall, the model is better at identifying profitable stores, which is the more business-critical prediction.

---
## Part 5 — Multiclass Classification: Predicting Expansion Tier (20 pts)

In [ ]:
exclude_mc = ['store_id', 'market_id', 'revenue_weekly', 'high_profit', 'expansion_tier']
X_mc = df_reg[[c for c in df_reg.columns if c not in exclude_mc]].astype(float)
y_mc = df['expansion_tier']

X_tr, X_te, y_tr, y_te = train_test_split(
    X_mc, y_mc, test_size=0.2, random_state=SEED
)

### 5.1 One-vs-Rest (OvR)

In [ ]:
ovr_model = LogisticRegression(multi_class='ovr', max_iter=1000, random_state=SEED)
ovr_model.fit(X_tr, y_tr)
y_pred_ovr = ovr_model.predict(X_te)

print('OvR Accuracy:', accuracy_score(y_te, y_pred_ovr))
print('\nOvR Confusion Matrix:')
print(confusion_matrix(y_te, y_pred_ovr, labels=['Go', 'Pilot', 'NoGo']))
print('\nOvR Classification Report:')
print(classification_report(y_te, y_pred_ovr))

### 5.2 One-vs-One (OvO)

In [ ]:
ovo_model = OneVsOneClassifier(LinearSVC(random_state=SEED, max_iter=2000))
ovo_model.fit(X_tr, y_tr)
y_pred_ovo = ovo_model.predict(X_te)

print('OvO Accuracy:', accuracy_score(y_te, y_pred_ovo))
print('\nOvO Confusion Matrix:')
print(confusion_matrix(y_te, y_pred_ovo, labels=['Go', 'Pilot', 'NoGo']))
print('\nOvO Classification Report:')
print(classification_report(y_te, y_pred_ovo))

### 5.3 Comparison

**OvR vs OvO:**
- **One-vs-Rest** trains one binary classifier per class (3 classifiers total), treating each class against all others combined. It is simpler and faster but can be imbalanced if one class dominates.
- **One-vs-One** trains one classifier per pair of classes (3 classifiers for 3 classes), then uses majority voting. It tends to handle class boundaries more precisely but requires more classifiers.

**Hardest class to predict:** The class with the lowest F1 score (likely `Pilot`) is hardest to predict, because it sits between `Go` and `NoGo` and may overlap with both in feature space.

**Why multiclass is harder:** Binary classification has a single decision boundary. Multiclass problems require the model to separate three distinct regions simultaneously, increasing complexity and the chance of misclassification at overlapping boundaries.

---
## Part 6 — Executive Recommendation (10 pts)

Based on the analyses conducted across all five parts, the following recommendations are provided to leadership:

**Campaign impact:** The independent samples t-test found a statistically significant difference in weekly revenue between stores that ran the marketing campaign and those that did not (p < 0.05), with campaign stores generating higher average revenue. The campaign appears to have been effective and should be extended to additional markets.

**Store format priority:** The one-way ANOVA and Tukey HSD post hoc analysis revealed that store formats differ significantly in mean weekly revenue. The Endcap format consistently shows the highest mean revenue and the strongest pairwise performance relative to other formats; future expansion should favor this format where feasible.

**Regression model:** The multiple linear regression model is statistically significant overall (F-test p < 0.05) and explains a meaningful proportion of variance in weekly revenue. Key drivers include foot traffic index, digital ad spend, and campaign participation, confirming that investment in traffic generation and advertising yields measurable revenue returns.

**Classifier recommendation:** The One-vs-One classifier with LinearSVC provides competitive accuracy for predicting expansion tier. It handles multi-class boundaries more carefully than OvR, making it preferable for the Go/Pilot/NoGo decision.

**Limitation:** The dataset is cross-sectional and observational; causal claims cannot be made with certainty. Confounding factors not captured in the dataset (e.g., local economic conditions, competitor promotions) may influence results.